# Publication figures

Consolidated, publication-quality figures for the digital-bat paper. Every analysis loads the **17 `DEFAULT_SESSIONS` once** (`use_kinematics=False`; Krishna sessions never appear) and uses the single consensus joint model `joint_v2_kin0_L4_d64_ff64_lam0.61_pw0.84_identity_we1.9_wd2.8_wa0.67_n17.pth` throughout. Shared style/stat/plot helpers live in `plotting_helpers.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sys.path.append(os.path.expanduser('~/Documents/code/utils'))
sys.path.append(os.path.expanduser('~/Documents/code/digitalbat'))

from digitalbat.utility.utils import (
    set_random_state, bat_palette, parse_bat, corr,
    poisson_fde_population, poisson_fde_population_filtered,
)
import notebook_helpers as nh
import plotting_helpers as ph

ph.set_pub_style()
ph.FIG_DIR = Path('out/figures')

seed = 1
set_random_state(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Shared setup

Load the 17 `DEFAULT_SESSIONS` **once** (`use_kinematics=False`, `behavior_dim==3`, Krishna excluded) and instantiate the single consensus joint model used throughout the paper. Downstream figures reuse these objects; nothing below re-loads data or re-instantiates the model.

In [ ]:
from digitalbat.datasets.data import bat_data_loader
from digitalbat.datasets.defaults import DEFAULT_SESSION_NAMES, make_data_config

# 17 DEFAULT_SESSIONS, position-only (use_kinematics=False -> behavior_dim == 3).
session_names = list(DEFAULT_SESSION_NAMES)
data_config = make_data_config(session_names=session_names)
session_names = data_config['session_names']
bin_width = data_config['bin_width']

dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)
assert behavior_dim == 3, 'figures.ipynb assumes position-only behaviour (use_kinematics=False)'

bats = ph.bat_order()
BAT_PALETTE = ph.bat_pal()
n_neurons_total = sum(unit_num_dict.values())
print(f'{len(session_names)} sessions | {n_neurons_total} neurons | behavior_dim={behavior_dim}')

In [ ]:
from digitalbat.models.joint import JointMaskedTransformer
from digitalbat.datasets.kinematics import make_joint_dataloaders

# --- Consensus model: the single joint encode/decode transformer used paper-wide.
JOINT_CKPT = ('./out/joint/joint_v2_kin0_L4_d64_ff64_lam0.61_pw0.84_identity'
              '_we1.9_wd2.8_wa0.67_n17.pth')
JOINT_PARAMS = dict(num_layers=4, latent_dim=64, num_heads=4, ff_dim=64, dropout=0.2)
INPUT_TRANSFORM = 'identity'


def joint_session_room(s):
    """3-config room map (matches joint-encode-decode.ipynb `session_room`)."""
    if '_fs' in s:
        return 'krishna_fs'
    if s.startswith('Dataset_2_'):
        return 'dataset_2'
    return 'lab'


session_to_room = {s: joint_session_room(s) for s in unit_num_dict}

set_random_state(seed)
joint_model = JointMaskedTransformer(
    unit_num_dict, behavior_dim=3, input_transform=INPUT_TRANSFORM,
    session_to_room=session_to_room, **JOINT_PARAMS).to(device)
joint_model.load_state_dict(torch.load(JOINT_CKPT, map_location=device))
joint_model = joint_model.eval()

# Joint position-only dataloaders + meta for the DECODE-arm figures (position from
# spikes). Kept separate from `dataloaders`/`meta_info_dict` (used by the ENCODE-arm
# figures) because make_joint_dataloaders can drop tail flights, shifting flight ids.
_dlp, _dlk, _, joint_meta = make_joint_dataloaders(data_config)
joint_loaders = _dlp
print('loaded consensus joint model:', JOINT_CKPT)

In [ ]:
# Encode-arm adapter: exposes the joint model with the Bat_Transformer interface
# (behaviour in -> spike rate out) plus per-session readouts, so the encoding /
# embedding / readout-weight figures can treat it like the standalone encoder.
class JointEncoderAdapter(nn.Module):
    def __init__(self, joint, unit_num_dict):
        super().__init__()
        self.joint = joint
        self.unit_num_dict = dict(unit_num_dict)
        self.readouts = {s: nn.Sequential(joint.spike_readout[s])
                         for s in joint.spike_readout}
        self.final_activation = joint.spike_act

    def forward(self, session_name, x, mask, return_attn_weights=False, **kwargs):
        B, T, _ = x.shape
        K = self.unit_num_dict[session_name]
        spike = x.new_zeros(B, T, K)
        nt = mask.to(x.dtype)            # encode: neural tokens are the targets
        bt = torch.zeros_like(nt)        # behaviour is seen
        if return_attn_weights:
            _, _, xfull, attn = self.joint(session_name, spike, x, mask, nt, bt,
                                           return_attn_weights=True)
            h_n = xfull[:, :T]
            rate = self.final_activation(self.readouts[session_name](h_n))
            return rate, [a[:, :, :T, T:] for a in attn]
        _, _, xfull = self.joint(session_name, spike, x, mask, nt, bt)
        h_n = xfull[:, :T]
        return self.final_activation(self.readouts[session_name](h_n))


bat_former = JointEncoderAdapter(joint_model, unit_num_dict).to(device).eval()


@torch.no_grad()
def encode_eval_records(model, loaders, meta, *, split='test'):
    """Per-(session, neuron) encode metrics on `split`: returns a tidy DataFrame
    with columns session, bat, unit, corr, fde (raw FDE, not clipped)."""
    rows = []
    for sn, sd in loaders[split].items():
        yv, yp = [], []
        for batch in sd:
            m = batch.mask.to(device)
            rate = model(sn, batch.behavior.to(device), m)
            yv.append(batch.spike[batch.mask == 1.0].numpy())
            yp.append(rate[m == 1.0].detach().cpu().numpy())
        yv, yp = np.vstack(yv), np.vstack(yp)
        c = corr(yv, yp, axis=0)
        f = poisson_fde_population(yv, yp)
        for u in range(yv.shape[1]):
            rows.append(dict(session=sn, bat=parse_bat(sn), unit=u,
                             corr=float(c[u]), fde=float(f[u])))
    return pd.DataFrame(rows)

---
## Figure 1 — Transformer encoding/decoding models outperform standard approaches

**a–b** Model schematics (omitted — drawn separately). **c** Predicted vs observed rate maps for two dual-field example units. **d–e** Per-neuron encoding fit (Poisson FDE, Pearson r) for GLM (place), GLM (trajectory-specific place), and the joint transformer. **f** Example decoded 3-D trajectory (joint). **g–h** Decoding accuracy (3-D Fréchet distance per flight; 3-D reconstruction error per point) for NB (place), unified CEBRA, and joint.

In [ ]:
# --- Joint DECODE arm: predict 3D position from spikes (all behaviour masked).
# Per-timestep records feed the trajectory panel (1f), reconstruction error (1h),
# and per-flight Fréchet distance (1g). Uses the joint decode loaders + meta.
from digitalbat.decoding import discrete_frechet

TRUTH_3D = ['x_true', 'y_true', 'z_true']
PRED_3D = ['x_pred', 'y_pred', 'z_pred']


@torch.no_grad()
def joint_decode_records(split='test'):
    recs = []
    for sn, sd in joint_loaders[split].items():
        bm = np.asarray(joint_meta['behavior_mean'][sn]).reshape(-1)[:3]
        bs = np.asarray(joint_meta['behavior_std'][sn]).reshape(-1)[:3]
        bat = parse_bat(sn)
        for batch in sd:
            m = batch.mask.to(device)
            bt = batch.mask.clone().to(device)      # decode: all behaviour masked
            nt = torch.zeros_like(bt)
            _, beh_pred, _ = joint_model(sn, batch.spike.to(device),
                                         batch.behavior.to(device), m, nt, bt)
            yp = beh_pred[:, :, :3].cpu().numpy() * bs + bm
            yt = batch.behavior[:, :, :3].numpy() * bs + bm
            mn = batch.mask.numpy().astype(bool)
            B, T = mn.shape
            fpi = np.broadcast_to(batch.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf = yt[mn], yp[mn]
            for i in range(len(ytf)):
                recs.append(dict(bat=bat, session=sn, flight_id=int(fpi[mn][i]),
                                 t_idx=int(tix[mn][i]),
                                 x_true=ytf[i, 0], y_true=ytf[i, 1], z_true=ytf[i, 2],
                                 x_pred=ypf[i, 0], y_pred=ypf[i, 1], z_pred=ypf[i, 2],
                                 err_2d=float(np.linalg.norm(ytf[i, :2] - ypf[i, :2])),
                                 err_3d=float(np.linalg.norm(ytf[i] - ypf[i]))))
    return pd.DataFrame(recs).assign(decoder='joint')


def frechet_per_flight(df, truth=TRUTH_3D, pred=PRED_3D):
    """Per-(decoder, session, flight) discrete 3D Fréchet distance."""
    rows = []
    gcols = ['bat', 'session', 'flight_id']
    gcols = (['decoder'] + gcols) if 'decoder' in df else gcols
    for keys, grp in df.groupby(gcols, sort=False, observed=True):
        grp = grp.sort_values('t_idx')
        row = dict(zip(gcols, keys if isinstance(keys, tuple) else (keys,)))
        row['frechet_3d'] = discrete_frechet(grp[truth].to_numpy(float),
                                             grp[pred].to_numpy(float))
        rows.append(row)
    return pd.DataFrame(rows)


joint_decode_test = joint_decode_records('test')
joint_frechet = frechet_per_flight(joint_decode_test)
print('joint decode: %d timepoints, %d flights' % (len(joint_decode_test), len(joint_frechet)))

In [ ]:
# === Fig 1c: dual-field example units (spikes / empirical rate / predicted rate) ===
# The paper's reference "Dataset_3_26659_240404 unit 148" was a typo for bat 29959;
# unit 1 is pinned to Dataset_3_29959_240404 unit 148 (peak sep ~6.6 m), unit 2 is
# auto-picked from a different bat. Edit FIG1C_PICKS to override.
FIG1C_PICKS = [('Dataset_3_29959_240404', 148)]   # pin unit 1; unit 2 auto (different bat)

from notebook_helpers import (build_flight_records, select_dual_field_cells,
                              plot_dual_field_cell, loc_bin_width_rm)

dt = bin_width
dual_candidates = []
for sn, sd in dataloaders['test'].items():
    Xl, yl, pl, ml, fl = [], [], [], [], []
    with torch.no_grad():
        for batch in sd:
            X = batch.behavior.to(device); m = batch.mask.to(device)
            rate = bat_former(sn, X, m)
            Xl.append(X.cpu().numpy()); yl.append(batch.spike.numpy())
            pl.append(rate.cpu().numpy()); ml.append(m.cpu().numpy())
            fl.append(batch.flight_type_index.numpy())
    Xa, ya, pa = np.vstack(Xl), np.vstack(yl), np.vstack(pl)
    ma, fa = np.vstack(ml), np.hstack(fl)
    yflat, pflat = ya[ma == 1.0], pa[ma == 1.0]
    ncorr = corr(yflat, pflat, axis=0)
    nfde = poisson_fde_population(yflat, pflat)
    r_lim = meta_info_dict['r_lim'][sn]
    flights = build_flight_records(Xa, ya, pa, ma, fa,
                                   meta_info_dict['behavior_mean'][sn],
                                   meta_info_dict['behavior_std'][sn])
    # n_top large so a specifically-requested unit is captured, not just the top 2.
    for c in select_dual_field_cells(flights, ya.shape[2], dt, r_lim,
                                     loc_bin_width=loc_bin_width_rm, n_top=100):
        dual_candidates.append(dict(session=sn, bat=parse_bat(sn), flights=flights,
                                    r_lim=r_lim, corr=float(ncorr[c['neuron']]),
                                    fde=float(nfde[c['neuron']]), **c))

by_key = {(d['session'], d['neuron']): d for d in dual_candidates}
cand_df = pd.DataFrame([{k: d[k] for k in ('session', 'bat', 'neuron', 'sep', 'corr', 'fde')}
                        for d in dual_candidates]).sort_values('sep', ascending=False)
print('Top dual-field candidates:'); print(cand_df.head(12).to_string(index=False))

dual_candidates.sort(key=lambda d: d['sep'], reverse=True)
picks = []
for p in (FIG1C_PICKS or []):
    if tuple(p) in by_key:
        picks.append(by_key[tuple(p)])
    else:
        print(f'WARNING: {p} not a dual-field candidate (thresholds); skipping pin.')
if not picks:                                   # fall back: best-separated cell
    picks = [dual_candidates[0]]
if len(picks) < 2:                              # add best cell from a different bat
    used = {pk['bat'] for pk in picks}
    picks.append(next((d for d in dual_candidates if d['bat'] not in used), dual_candidates[1]))

for pk in picks:
    title = (f"{pk['session']} — unit {pk['neuron']} | corr {pk['corr']:.2f}, "
             f"FDE {pk['fde']:.2f} | peak sep {pk['sep']:.1f} m")
    plot_dual_field_cell(pk['flights'], pk['neuron'], pk['clusters'], dt, pk['r_lim'],
                         title=title, peaks=pk['peaks'])
    plt.show()

In [ ]:
# === Fig 1d/e (+ ED Fig 1a/b): per-neuron encoding fit for GLM place, GLM
# trajectory-specific place, and the joint transformer ===
# Train-only Fréchet single-linkage flight clustering (alpha=0.8) supplies the
# trajectory-cluster place fields for the "joint"-conditioned GLM (matches
# model_comparison.ipynb / decode-position-comparison.ipynb).
from collections import Counter
from scipy.interpolate import interp1d
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from digitalbat.utility.utils import frechet_distance
from digitalbat.models.glm import (GLM_3DCosineBasis, GLM_Coupled,
                                   make_log_raised_cosine_basis)

CLUSTER_ALPHA, CLUSTER_DS, MIN_CLUSTER_FLIGHTS = 0.8, 7, 3
CLUSTER_MAX_ALPHA, UNCLUSTERED, N_LAGS = 1.5, -1, 10


def _downsample_traj(pos_xyz, mask, ds=CLUSTER_DS):
    out = np.full((pos_xyz.shape[0], ds, 3), np.nan)
    for i in range(pos_xyz.shape[0]):
        L = int(mask[i].sum())
        if L < 2:
            continue
        traj = pos_xyz[i, mask[i]]
        new_idx = np.linspace(0, L - 1, ds)
        for d in range(3):
            out[i, :, d] = interp1d(np.arange(L), traj[:, d])(new_idx)
    return out


def compute_flight_clusters(loaders, meta, alpha=CLUSTER_ALPHA):
    flight_cluster, cluster_keys = {}, {}
    for sn in loaders['train']:
        bm = np.asarray(meta['behavior_mean'][sn]).reshape(-1)
        bs = np.asarray(meta['behavior_std'][sn]).reshape(-1)
        trajs, fpis = {}, {}
        for split in ('train', 'valid', 'test'):
            ds = loaders[split][sn].dataset
            pos = (ds.behavior.numpy() * bs + bm)[..., :3]
            trajs[split] = _downsample_traj(pos, ds.mask.numpy().astype(bool))
            fpis[split] = ds.flight_path_index.numpy().astype(int)
        Xtr = trajs['train']
        keep = np.where(~np.isnan(Xtr[:, 0, 0]))[0]
        X, n = Xtr[keep], len(keep)
        mapping = {}
        if n < 2:
            for split in ('train', 'valid', 'test'):
                for f in fpis[split]:
                    mapping[int(f)] = UNCLUSTERED
            flight_cluster[sn], cluster_keys[sn] = mapping, []
            continue
        D = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                D[i, j] = D[j, i] = frechet_distance(X[i], X[j])
        used_alpha = alpha
        while True:
            Z = linkage(squareform(D, checks=False), method='single')
            labels = fcluster(Z, t=used_alpha, criterion='distance').astype(int)
            if len(np.unique(labels)) > 1 or used_alpha >= CLUSTER_MAX_ALPHA:
                break
            used_alpha += 0.1
        counts = Counter(labels)
        major = sorted(int(c) for c, k in counts.items() if k >= MIN_CLUSTER_FLIGHTS)
        majset = set(major)
        for p, gi in enumerate(keep):
            lab = int(labels[p])
            mapping[int(fpis['train'][gi])] = lab if lab in majset else UNCLUSTERED
        for gi in range(Xtr.shape[0]):
            mapping.setdefault(int(fpis['train'][gi]), UNCLUSTERED)
        for split in ('valid', 'test'):
            t = trajs[split]
            for i in range(t.shape[0]):
                f = int(fpis[split][i])
                if np.isnan(t[i, 0, 0]):
                    mapping[f] = UNCLUSTERED
                    continue
                dists = np.array([frechet_distance(t[i], X[j]) for j in range(n)])
                jm = int(np.argmin(dists))
                lab = int(labels[jm])
                mapping[f] = lab if (dists[jm] <= used_alpha and lab in majset) else UNCLUSTERED
        flight_cluster[sn], cluster_keys[sn] = mapping, major
    return flight_cluster, cluster_keys


def cluster_idx_from_fpi(fpi_array, session, flight_cluster):
    fmap = flight_cluster.get(session, {})
    return torch.tensor([fmap.get(int(f), UNCLUSTERED) for f in fpi_array], dtype=torch.long)


flight_cluster, cluster_keys_by_session = compute_flight_clusters(dataloaders, meta_info_dict)

# --- Build + load the two GLMs and evaluate per-neuron FDE/corr on test.
_glm_common = dict(n_bases_per_dim=16, pos_range=(-3.0, 3.0), width_to_spacing_ratio=4,
                   activation='exp', bin_width=bin_width)


@torch.no_grad()
def eval_glm_metrics(model, joint_cond=False):
    model = model.to(device).eval()
    rows = []
    is_coupled = isinstance(model, GLM_Coupled)
    for sn, sd in dataloaders['test'].items():
        yv, yp = [], []
        for b in sd:
            m = b.mask.to(device)
            ci = (cluster_idx_from_fpi(b.flight_path_index.numpy(), sn, flight_cluster).to(device)
                  if (is_coupled and joint_cond) else None)
            if is_coupled:
                out = model(sn, b.behavior.to(device), m, spike=b.spike.to(device), cluster_idx=ci)
            else:
                out = model(sn, b.behavior.to(device), m)
            v = m == 1.0
            yv.append(b.spike.to(device)[v].cpu().numpy()); yp.append(out[v].cpu().numpy())
        yv, yp = np.vstack(yv), np.vstack(yp)
        c, f = corr(yv, yp, axis=0), poisson_fde_population(yv, yp)
        for u in range(yv.shape[1]):
            rows.append(dict(session=sn, bat=parse_bat(sn), unit=u,
                             corr=float(c[u]), fde=float(f[u])))
    return pd.DataFrame(rows)


set_random_state(seed)
glm_place = GLM_3DCosineBasis(**_glm_common, output_dim_dict=unit_num_dict)
glm_place.load_state_dict(torch.load(
    './out/bat_glm/bat_glm_position_seed_1_optuna_acc50_default17.pth', map_location=device))
set_random_state(seed)
glm_traj = GLM_Coupled(output_dim_dict=unit_num_dict, **_glm_common, hist_basis=None,
                       coup_basis=None, acausal_basis=None, n_lags=N_LAGS,
                       cluster_keys=cluster_keys_by_session)
glm_traj.load_state_dict(torch.load(
    './out/bat_glm/bat_glm_joint_seed_1_optuna_acc50_default17.pth', map_location=device))

ENC_LABELS = {'glm_place': 'GLM (place)', 'glm_traj': 'GLM (traj. place)', 'joint': 'Joint'}
ENC_ORDER = ['glm_place', 'glm_traj', 'joint']
encode_metrics_df = pd.concat([
    eval_glm_metrics(glm_place, joint_cond=False).assign(model='glm_place'),
    eval_glm_metrics(glm_traj, joint_cond=True).assign(model='glm_traj'),
    encode_eval_records(bat_former, dataloaders, meta_info_dict).assign(model='joint'),
], ignore_index=True)
encode_metrics_df['model'] = pd.Categorical(encode_metrics_df['model'], ENC_ORDER, ordered=True)
print('encode metrics rows:', len(encode_metrics_df))

In [ ]:
# Fig 1d (FDE) and 1e (corr): session medians + per-neuron, stars vs Joint.
fig_d, ptab_d = ph.compare_models_fig(
    encode_metrics_df, 'fde', ENC_ORDER, ref='joint', labels=ENC_LABELS,
    ylabel='Poisson FDE', unit_name='neuron', clip=(-1, 1), zero_line=True)
fig_d.suptitle('Figure 1d — encoding fit (Poisson FDE)', y=1.03)
ph.save_fig(fig_d, 'fig1d_fde'); plt.show()

fig_e, ptab_e = ph.compare_models_fig(
    encode_metrics_df, 'corr', ENC_ORDER, ref='joint', labels=ENC_LABELS,
    ylabel='Pearson r', unit_name='neuron', clip=(-0.2, 1), zero_line=True)
fig_e.suptitle('Figure 1e — encoding fit (Pearson r)', y=1.03)
ph.save_fig(fig_e, 'fig1e_corr'); plt.show()

print('\nFig 1d — FDE, session-level Wilcoxon vs Joint:')
ph.p_table(ptab_d.to_dict('records'))
print('\nFig 1e — corr, session-level Wilcoxon vs Joint:')
ph.p_table(ptab_e.to_dict('records'))

In [ ]:
# === Fig 1f: example decoded 3D trajectory (joint decoder only) ===
from digitalbat.decoding import polylines_from_per_timestep_df, plot_trajectory_zoom_3d

# Pick an indoor session with many decoded flights for a clean zoom panel.
_counts = (joint_decode_test[joint_decode_test['session'].map(ph.is_indoor)]
           .groupby('session')['flight_id'].nunique())
zoom_session = _counts.idxmax() if len(_counts) else joint_decode_test['session'].iloc[0]

joint_polylines = polylines_from_per_timestep_df(
    joint_decode_test, pred_cols=('x_pred', 'y_pred', 'z_pred'),
    truth_cols=('x_true', 'y_true', 'z_true'), sort_by='t_idx')
fig_f = plot_trajectory_zoom_3d(
    zoom_session, [('joint', 'Joint (enc-dec)', 'tab:olive')],
    {'joint': joint_polylines}, figsize_per_panel=(5, 5),
    suptitle=f'Figure 1f — decoded vs true trajectories ({zoom_session})')
ph.save_fig(fig_f, 'fig1f_trajectory'); plt.show()

In [ ]:
# === Fig 1g/h baseline: naïve-Bayes (place) decoder ===
# Ported from decode-position-comparison.ipynb: per-session NB-2D/3D on the base
# dataloaders, hyperparameters from the cached Optuna studies.
from digitalbat.models.decoder import NaiveBayes_Decoder

nb_best = {}
for ndim in (2, 3):
    with open(f'./out/nb_decoder/nb_decoder_{ndim}d_seed_{seed}_optuna.pkl', 'rb') as fh:
        nb_best[ndim] = pickle.load(fh).best_params


def gather_session_arrays(loader, bm, bs):
    spk, beh, msk, fpi = [], [], [], []
    for batch in loader:
        spk.append(batch.spike.numpy()); beh.append(batch.behavior.numpy())
        msk.append(batch.mask.numpy().astype(bool)); fpi.append(batch.flight_path_index.numpy())
    spk = np.concatenate(spk); beh = np.concatenate(beh)
    msk = np.concatenate(msk); fpi = np.concatenate(fpi)
    bm_a, bs_a = np.asarray(bm).squeeze(), np.asarray(bs).squeeze()
    pos = beh[..., :3] * bs_a[None, None, :3] + bm_a[None, None, :3]
    B, T, _ = beh.shape
    fpit = np.broadcast_to(fpi[:, None], (B, T)); tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
    return dict(spikes=spk[msk], pos_x=pos[..., 0][msk], pos_y=pos[..., 1][msk],
                pos_z=pos[..., 2][msk], flight_id=fpit[msk].astype(int), t_idx=tix[msk].astype(int))


def make_nb(ndim, r_lim, p):
    n_bins = (p['n_bins_x'], p['n_bins_y'], p['n_bins_z']) if ndim == 3 else (p['n_bins_x'], p['n_bins_y'])
    return NaiveBayes_Decoder(n_bins=n_bins, lims=[tuple(r_lim[d]) for d in range(ndim)],
                              sigma=p['sigma'], dt=bin_width, min_occupancy=p['min_occupancy'],
                              use_uniform_prior=p['use_uniform_prior'])


def eval_nb_records():
    recs = []
    for sn in dataloaders['test']:
        bm, bs = meta_info_dict['behavior_mean'][sn], meta_info_dict['behavior_std'][sn]
        r_lim = meta_info_dict['r_lim'][sn]
        tr = gather_session_arrays(dataloaders['train'][sn], bm, bs)
        te = gather_session_arrays(dataloaders['test'][sn], bm, bs)
        if tr['spikes'].shape[0] < 100 or te['spikes'].shape[0] < 10:
            continue
        tr3 = np.column_stack([tr['pos_x'], tr['pos_y'], tr['pos_z']])
        te3 = np.column_stack([te['pos_x'], te['pos_y'], te['pos_z']])
        nb3 = make_nb(3, r_lim, nb_best[3]).fit(tr['spikes'], tr3)
        pd3, _ = nb3.predict(te['spikes'])
        err3 = np.sqrt(((te3 - pd3) ** 2).sum(1))
        for i in range(len(err3)):
            recs.append(dict(bat=parse_bat(sn), session=sn, flight_id=int(te['flight_id'][i]),
                             t_idx=int(te['t_idx'][i]), x_true=te3[i, 0], y_true=te3[i, 1],
                             z_true=te3[i, 2], x_pred=pd3[i, 0], y_pred=pd3[i, 1], z_pred=pd3[i, 2],
                             err_3d=float(err3[i])))
    return pd.DataFrame(recs)


nb_decode_test = eval_nb_records()
nb_frechet = frechet_per_flight(nb_decode_test)
print('NB (place) decode: %d points, %d flights' % (len(nb_decode_test), len(nb_frechet)))

In [ ]:
# === Fig 1g/h baseline: unified CEBRA decoder ===
# Ported from cebra-comparison.ipynb. Guarded: needs the source-only CEBRA repo on
# the path (~/Documents/code/CEBRA) and its cached unified solver; on any failure
# the panel simply omits CEBRA (a message is printed).
import hashlib
from sklearn.neighbors import KNeighborsRegressor

CEBRA_OK = True
try:
    sys.path.insert(0, os.path.expanduser('~/Documents/code/CEBRA'))
    import cebra, cebra.data, cebra.models, cebra.solver
    from cebra import CEBRA

    OUT_DIR, CEBRA_MAX_ITER, KNN_N, UNI_VARIANT = './out/cebra', 6000, 36, 'cebra_pos'
    CEBRA_HP = dict(model_architecture='offset10-model', learning_rate=3e-4, temperature=1.0,
                    time_offsets=10, output_dimension=3, num_hidden_units=32)
    UNIFIED_MASKS = {'RandomNeuronMask': (0.3, 0.9, 0.05), 'RandomTimestepMask': (0.3, 0.9, 0.05)}
    _sess_tag = hashlib.md5('|'.join(session_names).encode()).hexdigest()[:8]

    def _hp_tag():
        a = CEBRA_HP['model_architecture'].replace('-model', '')
        return (f"{a}_lr{CEBRA_HP['learning_rate']:.0e}_t{CEBRA_HP['temperature']:.2g}"
                f"_to{CEBRA_HP['time_offsets']}_d{CEBRA_HP['output_dimension']}_h{CEBRA_HP['num_hidden_units']}")

    def session_flights(session, split):
        bm = np.asarray(meta_info_dict['behavior_mean'][session]).squeeze()[:3]
        bs = np.asarray(meta_info_dict['behavior_std'][session]).squeeze()[:3]
        out = []
        for batch in dataloaders[split][session]:
            m = batch.mask.numpy().astype(bool)
            spike = batch.spike.numpy(); pos_z = batch.behavior[:, :, :3].numpy()
            fpi = batch.flight_path_index.numpy()
            for b in range(m.shape[0]):
                sel = m[b]
                if sel.sum() == 0:
                    continue
                out.append(dict(spike=spike[b, sel].astype(np.float32),
                                pos=(pos_z[b, sel] * bs[None] + bm[None]).astype(np.float32),
                                flight_id=int(fpi[b]), t_idx=np.flatnonzero(sel).astype(int)))
        return out

    def build_unified_solver(neuron_counts):
        model = cebra.models.init(CEBRA_HP['model_architecture'], num_neurons=int(sum(neuron_counts)),
                                  num_units=CEBRA_HP['num_hidden_units'],
                                  num_output=CEBRA_HP['output_dimension']).to(device)
        crit = cebra.models.FixedCosineInfoNCE(temperature=CEBRA_HP['temperature']).to(device)
        opt = torch.optim.Adam(list(model.parameters()), lr=CEBRA_HP['learning_rate'])
        return cebra.solver.UnifiedSolver(model=model, criterion=crit, optimizer=opt, tqdm_on=False)

    def fit_unified(Xs, Ys, path):
        solver = build_unified_solver([X.shape[1] for X in Xs])
        if os.path.exists(path):
            solver.load_state_dict(torch.load(path, map_location=device, weights_only=False), strict=True)
            return solver
        dsets = [cebra.data.TensorDataset(neural=Xs[i].astype(np.float32),
                                          continuous=Ys[i].astype(np.float32), device=device)
                 for i in range(len(Xs))]
        dataset = cebra.data.UnifiedDataset(dd for dd in dsets)
        dataset.set_masks(UNIFIED_MASKS); dataset.to(device)
        loader = cebra.data.UnifiedLoader(dataset, conditional='time_delta', num_steps=CEBRA_MAX_ITER,
                                          batch_size=512, time_offset=CEBRA_HP['time_offsets'])
        dataset.configure_for(solver.model)
        set_random_state(seed); solver.fit(loader)
        solver.save(logdir=os.path.dirname(path), filename=os.path.basename(path))
        return solver

    def unified_embed_split(solver, split):
        fb = {s: session_flights(s, split) for s in session_names}
        Xs_all = [np.vstack([f['spike'] for f in fb[s]]).astype(np.float32) for s in session_names]
        Ys_all = [np.vstack([f['pos'] for f in fb[s]]).astype(np.float32) for s in session_names]
        Xt, Yt = [torch.from_numpy(X) for X in Xs_all], [torch.from_numpy(Y) for Y in Ys_all]
        solver = solver.to('cpu'); out = {}
        for i, s in enumerate(session_names):
            bs_ = min(512, max(2, len(Xs_all[i]) // 3))
            emb = solver.transform(Xt, Yt, session_id=i, batch_size=bs_).cpu().numpy()
            pieces, c = [], 0
            for f in fb[s]:
                L = len(f['spike']); pieces.append((emb[c:c + L], f)); c += L
            out[s] = pieces
        solver.to(device)
        return out

    _train_by = {s: session_flights(s, 'train') for s in session_names}
    Xs = [np.vstack([f['spike'] for f in _train_by[s]]).astype(np.float32) for s in session_names]
    Ys = [np.vstack([f['pos'] for f in _train_by[s]]).astype(np.float32) for s in session_names]
    unified_path = f'{OUT_DIR}/unified_{UNI_VARIANT}__{_hp_tag()}__it{CEBRA_MAX_ITER}__seed{seed}__s{len(session_names)}_{_sess_tag}.pt'
    unified_model = fit_unified(Xs, Ys, unified_path)

    uni_tr, uni_te = unified_embed_split(unified_model, 'train'), unified_embed_split(unified_model, 'test')
    crecs = []
    for s in session_names:
        Etr = np.vstack([e for e, _ in uni_tr[s]]); Ytr = np.vstack([f['pos'] for _, f in uni_tr[s]])
        knn = KNeighborsRegressor(n_neighbors=KNN_N, metric='cosine').fit(Etr, Ytr)
        for emb, f in uni_te[s]:
            pred, true = np.asarray(knn.predict(emb)), f['pos']
            e3 = np.sqrt(((true - pred) ** 2).sum(1))
            for i in range(len(true)):
                crecs.append(dict(bat=parse_bat(s), session=s, flight_id=f['flight_id'],
                                  t_idx=int(f['t_idx'][i]), x_true=true[i, 0], y_true=true[i, 1],
                                  z_true=true[i, 2], x_pred=pred[i, 0], y_pred=pred[i, 1],
                                  z_pred=pred[i, 2], err_3d=float(e3[i])))
    cebra_decode_test = pd.DataFrame(crecs)
    cebra_frechet = frechet_per_flight(cebra_decode_test)
    print('unified CEBRA decode: %d points, %d flights' % (len(cebra_decode_test), len(cebra_frechet)))
except Exception as exc:
    CEBRA_OK = False
    cebra_decode_test = cebra_frechet = None
    print(f'[skip CEBRA] {type(exc).__name__}: {exc}')

In [ ]:
# === Fig 1g/h: decoding accuracy for NB (place), unified CEBRA, joint ===
DECODE_LABELS = {'naive_bayes': 'NB (place)', 'cebra': 'unified CEBRA', 'joint': 'Joint'}
DECODE_ORDER = ['naive_bayes', 'cebra', 'joint']

# Assemble per-flight Fréchet (1g) and per-point recon error (1h) across decoders.
_fr = [joint_frechet.assign(decoder='joint'), nb_frechet.assign(decoder='naive_bayes')]
_rc = [joint_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='joint'),
       nb_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='naive_bayes')]
if cebra_frechet is not None:
    _fr.append(cebra_frechet.assign(decoder='cebra'))
    _rc.append(cebra_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='cebra'))

decode_frechet_df = pd.concat(_fr, ignore_index=True)
decode_recon_df = pd.concat(_rc, ignore_index=True)
present = [d for d in DECODE_ORDER if d in set(decode_frechet_df['decoder'])]
for df in (decode_frechet_df, decode_recon_df):
    df['decoder'] = pd.Categorical(df['decoder'], present, ordered=True)

fig_g, ptab_g = ph.compare_models_fig(
    decode_frechet_df, 'frechet_3d', present, ref='joint', labels=DECODE_LABELS,
    model_col='decoder', ylabel='3D Fréchet (m)', unit_name='flight', higher_is_better=False)
fig_g.suptitle('Figure 1g — decoding: 3D Fréchet distance', y=1.03)
ph.save_fig(fig_g, 'fig1g_frechet'); plt.show()

fig_h, ptab_h = ph.compare_models_fig(
    decode_recon_df, 'err_3d', present, ref='joint', labels=DECODE_LABELS,
    model_col='decoder', ylabel='3D recon error (m)', unit_name='point', higher_is_better=False)
fig_h.suptitle('Figure 1h — decoding: 3D reconstruction error', y=1.03)
ph.save_fig(fig_h, 'fig1h_recon'); plt.show()

print('\nFig 1g — 3D Fréchet, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_g.to_dict('records'))
print('\nFig 1h — 3D recon error, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_h.to_dict('records'))

**Figure 1 | Transformer-based encoding/decoding models outperform standard approaches on hippocampal data.**
**a–b**, Model schematics (masking scheme; dual-branch architecture), drawn separately.
**c**, Two example units with spatially separated place fields on two different flight routes. For each unit and route: observed spikes (red, area ∝ count) on the flight path, the empirical firing-rate map, and the joint model's predicted-rate map (empirical and predicted share a colour scale).
**d–e**, Encoding fit quantified as per-neuron Poisson fraction deviance explained (**d**) and Pearson correlation (**e**) between predicted and observed spike trains on held-out flights, for a place GLM, a trajectory-specific-place GLM, and the joint transformer. Left, session medians (points = sessions, coloured by bat); right, all neurons pooled. Boxes show median and IQR; whiskers 1.5× IQR.
**f**, Representative decoded (olive) versus true (black) 3-D flight trajectories from the joint decoder for one session.
**g–h**, Decoding accuracy as per-flight 3-D discrete Fréchet distance (**g**) and per-point 3-D reconstruction error (**h**) for a naïve-Bayes place decoder, unified CEBRA, and the joint decoder.
Stars denote two-sided Wilcoxon signed-rank tests on paired per-session medians versus the joint model (*P* < 0.05 *, *P* < 0.01 **, *P* < 0.001 ***); n.s. comparisons are unmarked. Exact *P* values are printed in the tables above each panel. FDE axes are clipped to [−1, 1] for display; all statistics use raw values.

---
## Figure 2 — Transformer latents warp physical space to emphasise route identity

Physical XYZ, the 64-d transformer latent (PCA→3D), and the unified-CEBRA latent, each coloured by **x-coordinate** (top row) and by **flight route** (bottom row). Restricted to the indoor lab room (Datasets 1 & 3). The latent-extraction cell below is reused by Figure 4.

In [ ]:
# --- Shared latent extraction (encode pass): per-timepoint neural latent h_n,
# predicted rate, true position, and per-flight route label. Reused by Fig 4.
from notebook_helpers import DIST_THRESH, get_target_locs


@torch.no_grad()
def extract_latents(split='test'):
    """Concatenate encode-pass latents + metadata across all sessions.

    Returns a dict with arrays aligned per valid timepoint: `latent` (N, d),
    `pred` (N, K_session — ragged, so returned per-session too), `xyz` (N, 3),
    `session`, `bat`, `flight_id`, `rel_t`, and per-point `route`/`origin`/`dest`.
    """
    lat, xyz, sess, batv, fid, relt = [], [], [], [], [], []
    per_session_latent = {}
    for sn in session_names:
        bm = meta_info_dict['behavior_mean'][sn][0, 0, :3]
        bs = meta_info_dict['behavior_std'][sn][0, 0, :3]
        sl = []
        for batch in dataloaders[split][sn]:
            X = batch.behavior.to(device)
            m = batch.mask.to(device)
            B, T, _ = X.shape
            spike0 = X.new_zeros(B, T, unit_num_dict[sn])
            _, _, xf = joint_model(sn, spike0, X, m, m, torch.zeros_like(m))
            h_n = xf[:, :T].cpu().numpy()
            Xn = batch.behavior.numpy()
            mn = batch.mask.numpy() == 1.0
            fpi = batch.flight_path_index.numpy()
            for b in range(B):
                v = mn[b]
                nv = int(v.sum())
                if nv == 0:
                    continue
                lat.append(h_n[b, v]); sl.append(h_n[b, v])
                pos = Xn[b, :, :3] * bs + bm
                xyz.append(pos[v])
                sess.append(np.full(nv, sn)); batv.append(np.full(nv, parse_bat(sn)))
                fid.append(np.full(nv, int(fpi[b])))
                relt.append(np.arange(nv) / max(nv - 1, 1))
        per_session_latent[sn] = np.concatenate(sl) if sl else np.empty((0, joint_model.latent_dim))
    out = dict(latent=np.concatenate(lat), xyz=np.vstack(xyz),
               session=np.concatenate(sess), bat=np.concatenate(batv),
               flight_id=np.concatenate(fid), rel_t=np.concatenate(relt),
               per_session_latent=per_session_latent)

    # Per-(session, flight) origin/destination endpoint classes -> route id.
    origin = np.zeros(len(out['session']), int)
    dest = np.zeros_like(origin)
    key = np.char.add(np.char.add(out['session'], '|'), out['flight_id'].astype(str))
    for k in np.unique(key):
        m = key == k
        sn = out['session'][m][0]
        rew, perch = get_target_locs(sn)
        order = np.argsort(out['rel_t'][m])
        pos = out['xyz'][m][order]
        o = ph.endpoint_class(pos[0], rew, perch, DIST_THRESH)
        d = ph.endpoint_class(pos[-1], rew, perch, DIST_THRESH)
        origin[m] = o; dest[m] = d
    out['origin'] = origin
    out['dest'] = dest
    out['route'] = np.array([ph.route_id(o, d) for o, d in zip(origin, dest)])
    return out


emb = extract_latents('test')
print('latent points:', len(emb['session']), '| latent dim:', emb['latent'].shape[1])

In [ ]:
# === Figure 2 a-f: physical / transformer-latent / CEBRA embeddings (indoor only) ===
# Rows: colour by x-coordinate (a-c) and by flight route (d-f).
rng = np.random.default_rng(0)
ind = np.where(np.array([ph.is_indoor(s) for s in emb['session']]))[0]
if len(ind) > 40000:
    ind = np.sort(rng.choice(ind, 40000, replace=False))
xyz_i, sess_i, route_i = emb['xyz'][ind], emb['session'][ind], emb['route'][ind]
colx_i = ph.zscore_by_session(xyz_i[:, 0], sess_i)

# Physical (top-down x-y) and transformer latent (64d -> lobe-rotated 2D).
lat2d, _lat_tf = ph.lobe_projection(emb['latent'][ind], route_i)
panels = [('Physical (x–y)', xyz_i[:, :2], colx_i, route_i),
          ('Transformer latent', lat2d, colx_i, route_i)]

# Unified CEBRA embedding for the same indoor test flights (guarded on CEBRA).
if cebra_frechet is not None:
    ce, cx, croute = [], [], []
    for s in session_names:
        if not ph.is_indoor(s):
            continue
        rew, perch = get_target_locs(s)
        for embp, f in uni_te[s]:
            pos = f['pos']
            o = ph.endpoint_class(pos[0], rew, perch, DIST_THRESH)
            d = ph.endpoint_class(pos[-1], rew, perch, DIST_THRESH)
            ce.append(embp); cx.append(pos[:, 0])
            croute.append(np.full(len(embp), ph.route_id(o, d)))
    ce = np.vstack(ce); croute = np.concatenate(croute)
    cx = ph.zscore_by_session(np.concatenate(cx),
                              np.concatenate([np.full(len(e), s) for s in session_names
                                              if ph.is_indoor(s) for e, _ in uni_te[s]]))
    cebra2d, _ = ph.lobe_projection(ce, croute)
    panels.append(('Unified CEBRA', cebra2d, cx, croute))

fig2, axes = plt.subplots(2, len(panels), figsize=(4.2 * len(panels), 8), squeeze=False)
xvmin, xvmax = np.percentile(colx_i, [2, 98])
for j, (title, e2, cxj, rj) in enumerate(panels):
    sc = ph.plot_embedding(axes[0, j], e2, cxj, cmap='viridis', vlim=(xvmin, xvmax), s=3)
    axes[0, j].set_title(title)
    ph.plot_embedding(axes[1, j], e2, rj, categorical=True, s=3)
axes[0, 0].set_ylabel('coloured by x')
axes[1, 0].set_ylabel('coloured by route')
fig2.colorbar(sc, ax=axes[0, :].tolist(), fraction=0.02, pad=0.01, label='x (z-scored)')
handles = [plt.Line2D([], [], marker='o', ls='', color=ph.ROUTE_COLORS[k], label=ph.ROUTE_LABELS[k])
           for k in range(len(ph.ROUTE_LABELS))]
axes[1, -1].legend(handles=handles, title='route', loc='center left',
                   bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
fig2.suptitle('Figure 2 — physical vs latent vs CEBRA embeddings (indoor sessions)', y=1.01)
ph.save_fig(fig2, 'fig2_embeddings'); plt.show()

**Figure 2 | The transformer latent embedding warps physical space to emphasise flight-route identity and align across animals.** Each point is one held-out flight time-bin from the indoor lab sessions (Datasets 1 & 3). Columns: true 3-D position (top-down x–y), the 64-dimensional transformer latent projected to 2-D (PCA followed by a rotation that separates the two dominant route lobes horizontally), and the unified-CEBRA latent (same rotation procedure). **a–c**, coloured by (per-session z-scored) x-coordinate. **d–f**, coloured by flight route (origin→destination endpoint class). Unlike physical space and the CEBRA embedding, the transformer latent segregates flights primarily by route identity rather than by instantaneous position.